# FULL and MASKED length statistics

**Scope:** 81 training cases (2010–2020) and 16 development cases (2021–2022),
97 combined. FULL v1 and MASKED v2 use the same case IDs. The 16 final-test cases
and pending Sodhi case are excluded from tokenization and these summaries, per
Yinkui's handoff. This is descriptive measurement, not model training.

**Tokenizer:** `nlpaueb/legal-bert-base-uncased`, revision
`15b570cbf88259610b082a167dacc190124f60f6`, slow `BertTokenizer`, Transformers
4.57.1. Count untruncated content tokens **before CLS/SEP and padding**. A single
LEGAL-BERT input holds 510 content tokens plus two special tokens. Chunks are
contiguous 510-token windows without overlap, retaining the final short chunk.

Words are whitespace-separated, consistent with the preprocessing audit. P90
uses linear interpolation at `0.9 × (n − 1)` in the sorted observations.
All percentages show their case or word/token denominator explicitly.

Run with the MVP `.venv` Python kernel. To prepare another environment, install
`preprocessing/length_requirements-lock.txt` and run
`preprocessing/download_length_tokenizer.py` once. Subsequent execution is offline.


In [1]:
from pathlib import Path
import sys, json, html
from IPython.display import display, HTML
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
sys.path.insert(0, str(MVP / "preprocessing"))
import length_statistics as lengths
# Small independent checks for interpolation and chunk boundaries.
assert lengths.describe([10, 20, 30, 40])["p90"] == 37
assert lengths.describe([7]) == dict(minimum=7, median=7, mean=7, p90=7, maximum=7)
assert lengths.chunk_lengths(0) == []
assert lengths.chunk_lengths(510) == [510]
assert lengths.chunk_lengths(511) == [510, 1]
assert lengths.chunk_lengths(1020) == [510, 510]
assert lengths.chunk_lengths(1100) == [510, 510, 80]
report, per_case, paired = lengths.build(ROOT)

def table(headers, rows):
    def fmt(value):
        return f"{value:,.2f}" if isinstance(value, float) else str(value)
    body = "".join("<tr>" + "".join("<td>" + html.escape(fmt(v)) + "</td>" for v in row) + "</tr>" for row in rows)
    display(HTML("<style>.lengths{border-collapse:collapse;font-variant-numeric:tabular-nums}.lengths td,.lengths th{padding:7px 12px;border:1px solid #ddd;text-align:right}.lengths th{background:#f1f5f9}</style><table class='lengths'><thead><tr>" + "".join("<th>" + html.escape(h) + "</th>" for h in headers) + "</tr></thead><tbody>" + body + "</tbody></table>"))
print("Validated:", len(per_case), "case/variant rows;", len(paired), "paired cases.")


/Users/rebeccaweng/.codex/.chatgpt-projects/g-p-6ad5286760b08191b7b2578f60e435e7/legal-judgment/annotation/submissions/becca/MVP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Validated: 194 case/variant rows; 97 paired cases.


## Word and token lengths

Each row summarizes complete opinions in the named split/variant. Combined
statistics pool all 97 cases rather than averaging split-level statistics.
Content-token counts are checkpoint-specific; word counts are a separate measure.


In [2]:
stats = ("minimum", "median", "mean", "p90", "maximum")
table(["Scope", "Variant", "Measure", "Cases", "Minimum", "Median", "Mean", "P90", "Maximum"],
      [[s["scope"], s["variant"], metric, s["cases"], *[s[metric][k] for k in stats]]
       for s in report["summary"] for metric in ("words", "content_tokens")])


Scope,Variant,Measure,Cases,Minimum,Median,Mean,P90,Maximum
train,FULL,words,81,149,468,514.19,830.00,1486
train,FULL,content_tokens,81,223,641,705.62,"1,162.00",1878
train,MASKED,words,81,64,428,471.96,771.00,1425
train,MASKED,content_tokens,81,82,593,649.69,"1,083.00",1788
development,FULL,words,16,276,534.00,650.88,925.50,1796
development,FULL,content_tokens,16,375,714.00,868.25,"1,252.00",2320
development,MASKED,words,16,250,507.50,596.06,872.00,1671
development,MASKED,content_tokens,16,331,679.00,796.12,"1,208.00",2147
combined,FULL,words,97,149,486,536.73,846.80,1796
combined,FULL,content_tokens,97,223,664,732.44,"1,184.00",2320


## LEGAL-BERT input limit and chunks per case

“Exceeding” means **more than 510 content tokens**, not 512 content tokens.
The percentage denominator is the number of cases in that row. Chunk counts
include the final short chunk. No content is truncated or duplicated.


In [3]:
table(["Scope", "Variant", "Cases", "Exceeding", "% exceeding", "Total chunks", "Min/case", "Median/case", "Mean/case", "P90/case", "Max/case"],
      [[s["scope"], s["variant"], s["cases"], s["exceeding_one_input"],
        100*s["fraction_exceeding_one_input"], s["total_chunks"],
        *[s["chunks"][k] for k in stats]] for s in report["summary"]])


Scope,Variant,Cases,Exceeding,% exceeding,Total chunks,Min/case,Median/case,Mean/case,P90/case,Max/case
train,FULL,81,54,66.67,147,1,2,1.81,3.00,4
train,MASKED,81,50,61.73,142,1,2,1.75,3.00,4
development,FULL,16,14,87.50,36,1,2.00,2.25,3.00,5
development,MASKED,16,13,81.25,34,1,2.00,2.12,3.00,5
combined,FULL,97,68,70.10,183,1,2,1.89,3.00,5
combined,MASKED,97,63,64.95,176,1,2,1.81,3.00,5


## Masking-related length changes

Positive reductions mean FULL minus MASKED; negative values, if present, mean
an increase. “Overall % removed” is total reduction divided by the FULL total.
“Mean case %” averages each case's percentage; the two measures need not agree.
The reduction distribution reports paired differences, not differences between
independently summarized medians or percentiles.


In [4]:
table(["Scope", "Measure", "FULL total", "MASKED total", "Removed", "Overall % removed", "Mean case % removed"],
      [[s["scope"], metric, s[metric]["full_total"], s[metric]["masked_total"],
        s[metric]["total_reduction"], 100*s[metric]["corpus_reduction_fraction"],
        100*s[metric]["fractional_reduction"]["mean"]]
       for s in report["masking_changes"] for metric in ("words", "content_tokens", "chunks")])
table(["Scope", "Measure removed per case", "Minimum", "Median", "Mean", "P90", "Maximum"],
      [[s["scope"], metric, *[s[metric]["absolute_reduction"][k] for k in stats]]
       for s in report["masking_changes"] for metric in ("words", "content_tokens", "chunks")])
table(["Scope", "Cases moving from >1 input to 1 input after masking"],
      [[s["scope"], s["cases_crossing_to_one_input"]] for s in report["masking_changes"]])


Scope,Measure,FULL total,MASKED total,Removed,Overall % removed,Mean case % removed
train,words,41649,38229,3420,8.21,9.86
train,content_tokens,57155,52625,4530,7.93,9.57
train,chunks,147,142,5,3.40,2.78
development,words,10414,9537,877,8.42,8.54
development,content_tokens,13892,12738,1154,8.31,8.39
development,chunks,36,34,2,5.56,5.21
combined,words,52063,47766,4297,8.25,9.64
combined,content_tokens,71047,65363,5684,8.00,9.38
combined,chunks,183,176,7,3.83,3.18


Scope,Measure removed per case,Minimum,Median,Mean,P90,Maximum
train,words,8,39,42.22,66.00,145
train,content_tokens,10,49,55.93,90.00,176
train,chunks,0,0,0.06,0.00,1
development,words,12,38.50,54.81,113.50,161
development,content_tokens,14,46.50,72.12,155.00,222
development,chunks,0,0.00,0.12,0.50,1
combined,words,8,39,44.30,69.00,161
combined,content_tokens,10,49,58.60,94.20,222
combined,chunks,0,0,0.07,0.00,1


Scope,Cases moving from >1 input to 1 input after masking
train,4
development,1
combined,5


## Per-case outputs and validation

`data/length_statistics_v1/per_case.jsonl` contains all 194 rows, including each
text's SHA-256, words, content tokens, chunk count, chunk lengths and final-chunk
length. `paired_changes.jsonl` stores all 97 paired changes. `summary.json` stores
unrounded aggregate statistics; `manifest.json` pins text, code, tokenizer files,
software, split assignments and encoder settings. The preview below shows ten
rows; all case/variant rows are saved in the outputs.

Every chunk is checked to contain 1–510 tokens and at most 512 tokens after
CLS/SEP. Chunk lengths sum to complete token counts. Word reductions match the
MASKED v2 audit. Empty outputs fail explicitly; near-empty outputs are reported.
These checks do not resolve masking flags, family uncertainty or provisional
labels. New data or tokenizer/chunk settings require a new statistics version and
rerunning affected experiments. Original text, labels and splits are unchanged.


In [5]:
table(["Case", "Split", "Variant", "Words", "Content tokens", "Chunks", "Final chunk tokens"],
      [[r[k] for k in ("case_id", "split", "variant", "words", "content_tokens", "chunks", "final_chunk_content_tokens")]
       for r in per_case[:10]])
print("Empty outputs:", report["empty_word_or_token_outputs"])
print("Near-empty outputs (<50 words):", report["near_empty_under_50_words"])
print("All chunk checks passed:", report["all_chunk_checks_passed"])
print("Test/pending tokenization: excluded")
print("Saved outputs:", (MVP / "data/length_statistics_v1").relative_to(ROOT))


Case,Split,Variant,Words,Content tokens,Chunks,Final chunk tokens
NY-2682947,train,FULL,1486,1878,4,348
NY-2682947,train,MASKED,1425,1788,4,258
NY-2686388,train,FULL,320,462,1,462
NY-2686388,train,MASKED,279,413,1,413
NY-2686408,train,FULL,403,547,2,37
NY-2686408,train,MASKED,380,518,2,8
NY-2719389,train,FULL,414,532,2,22
NY-2719389,train,MASKED,391,505,1,505
NY-2757329,train,FULL,472,642,2,132
NY-2757329,train,MASKED,433,593,2,83


Empty outputs: []
Near-empty outputs (<50 words): []
All chunk checks passed: True
Test/pending tokenization: excluded
Saved outputs: annotation/submissions/becca/MVP/data/length_statistics_v1
